In [ ]:
!pip install -q --upgrade openai

In [ ]:
!pip install gradio

In [ ]:
import os
from IPython.display import Markdown, display
from openai import OpenAI
from google.colab import userdata
api = userdata.get('openaiapi')

In [ ]:
open_ai_client = OpenAI(api_key=api, base_url = "https://openrouter.ai/api/v1")

In [ ]:
def print_markdown(message):
  display(Markdown(message))

In [ ]:
def get_ai_tutor_response(user_question):
  system_prompt = "Your an AI Tutor that explain concepts clearly and concisely"
  try:
    response = open_ai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages = [
            {"role":"system", "content":system_prompt},
            {"role":"user", "content":user_question}
        ]
    )
    return response.choices[0].message.content
  except Exception as e:
    print(e)
    return f"Sorry, I encountered an error trying to get an answer: {e}"

In [ ]:
test_question = "Explain what a neural network is"
response = get_ai_tutor_response(test_question)
print_markdown(response)

In [ ]:
import gradio as gr

In [ ]:
expereince_level= {
    1: "like I'm 5 years old",
    2: "like I'm 10 years old",
    3: "like a high school student",
    4: "like a college student",
    5: "like an expert in the field",
}
def stream_ai_response_experience_level(user_question,expereincelevel):


  level = expereince_level.get(expereincelevel, "in a clear and concise way")
  system_prompt = f"You are a helpful AI Tutor. Explain the following concept {level}"
  try:
    stream = open_ai_client.chat.completions.create(
      model="gpt-4o-mini",
      messages = [
          {"role":"system", "content":system_prompt},
          {"role":"user", "content":user_question}
      ],
      stream=True
  )
    text_chunk = ""
    for chunk in stream:
      if chunk.choices[0].delta and chunk.choices[0].delta.content:
        text_chunk += chunk.choices[0].delta.content
        yield text_chunk

  except Exception as e:
    print(e)
    yield f"Sorry, I encountered an error trying to get an answer: {e}"



In [ ]:
ai_tutor_interface = gr.Interface(fn=stream_ai_response_experience_level,
                                  inputs=[gr.Textbox(lines=2, placeholder="Ask me anything...", label="Your Question"),
                                          gr.Slider(minimum=1, maximum=5, step=1,value=3, label="Experience Level")],
                                  outputs=gr.Textbox(lines=2, label="AI Tutor Response", container = True),
                                  title="Your best AI Tutor",
                                  description= "Enter your question below and the AI Tutor will provide an explanation.",
                                  flagging_mode="never")
ai_tutor_interface.launch(share=True)